# FITE Anonymized Classification Project


## Environment setup

Run the commands in `README.md` once from the VS Code terminal. If you need to install packages directly from the notebook, uncomment and run:

```python
# %pip install -r ../requirements.txt
```

In [ ]:
from pathlib import Path
import os
import subprocess


def find_project_root(start=None):
    """Find the project root from a notebook or script location."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start] + list(start.parents)
    for path in candidates:
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    return start


PROJECT_DIR = find_project_root()
os.chdir(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data" / "raw"
SUBMISSION_DIR = PROJECT_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / "train_data.csv"
test_path = DATA_DIR / "test_data.csv"
sample_path = DATA_DIR / "sample_submission.csv"

print("Project directory:", PROJECT_DIR)
print("Data directory   :", DATA_DIR)
print("Train exists     :", train_path.exists())
print("Test exists      :", test_path.exists())
print("Sample exists    :", sample_path.exists())
print("DVC file exists  :", (PROJECT_DIR / "data" / "raw.dvc").exists())

## DVC and Git metadata

This cell records the current Git commit and the DVC pointer file path. These values are later logged inside MLflow so each run is tied to a specific code version and data version.

In [ ]:
def safe_shell(command, cwd=PROJECT_DIR):
    try:
        return subprocess.check_output(command, shell=True, cwd=str(cwd), text=True).strip()
    except Exception:
        return "not_available"

GIT_COMMIT = safe_shell("git rev-parse HEAD")
DVC_DATA_POINTER = PROJECT_DIR / "data" / "raw.dvc"

print("Git commit:", GIT_COMMIT)
print("DVC pointer:", DVC_DATA_POINTER)
print("DVC pointer exists:", DVC_DATA_POINTER.exists())

## Important note about test data

The uploaded files included `train_data.csv` and `sample_submission.csv`. The project is ready for training and MLflow logging now, but Kaggle submission generation requires `test_data.csv`. Copy the competition `test_data.csv` into `data/raw/` before running the final submission cell.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)

In [ ]:
if not train_path.exists():
    raise FileNotFoundError(f"train_data.csv not found at: {train_path}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path) if test_path.exists() else None
sample_submission = pd.read_csv(sample_path) if sample_path.exists() else None

print("Train shape:", train.shape)
print("Test shape :", None if test is None else test.shape)
print("Sample submission shape:", None if sample_submission is None else sample_submission.shape)

if test is None:
    print("NOTE: test_data.csv is missing. EDA/CV/MLflow will run, but submission creation will be skipped until you add test_data.csv to data/raw/.")

In [ ]:
TARGET_COL = "target"
ID_COL = "ID"
feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET_COL]]

print("Columns:")
print(list(train.columns))
print("\nNumber of features:", len(feature_cols))
print("Feature columns:", feature_cols)

if test is not None:
    missing_in_test = set(feature_cols) - set(test.columns)
    extra_in_test = set(test.columns) - set(feature_cols) - {ID_COL}
    print("\nMissing features in test:", missing_in_test)
    print("Extra columns in test:", extra_in_test)

In [ ]:
display(train.head(10))
print("\nInfo:")
train.info()

In [ ]:
target_counts = train[TARGET_COL].value_counts()
target_percent = train[TARGET_COL].value_counts(normalize=True).mul(100).round(2)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_percent
})
display(target_summary)

plt.figure(figsize=(7, 4))
target_counts.plot(kind="bar")
plt.title("Target distribution")
plt.xlabel("target")
plt.ylabel("count")
plt.show()

majority_ratio = target_counts.max() / target_counts.sum()
print(f"Majority class ratio: {majority_ratio:.4f}")

In [ ]:
missing_train = train[feature_cols].isna().sum().sort_values(ascending=False)
missing_percent = (missing_train / len(train) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing_train, "missing_percent": missing_percent})

display(missing_table[missing_table["missing_count"] > 0])
print("Total missing in train features:", int(missing_train.sum()))

if test is not None:
    missing_test = test[feature_cols].isna().sum().sort_values(ascending=False)
    missing_test_percent = (missing_test / len(test) * 100).round(2)
    missing_test_table = pd.DataFrame({"missing_count": missing_test, "missing_percent": missing_test_percent})
    print("\nTest missing values:")
    display(missing_test_table[missing_test_table["missing_count"] > 0])
    print("Total missing in test features:", int(missing_test.sum()))

In [ ]:
duplicated_ids = train[ID_COL].duplicated().sum()
full_duplicate_rows = train.duplicated().sum()
duplicate_feature_rows = train.duplicated(subset=feature_cols).sum()
duplicate_feature_target_rows = train.duplicated(subset=feature_cols + [TARGET_COL]).sum()

print("Number of rows:", len(train))
print("Unique IDs:", train[ID_COL].nunique())
print("Duplicated IDs:", duplicated_ids)
print("\nFull duplicate rows:", full_duplicate_rows)
print("Duplicate feature rows ignoring ID and target:", duplicate_feature_rows)
print("Duplicate feature+target rows ignoring ID:", duplicate_feature_target_rows)

feature_group_sizes = train.groupby(feature_cols, dropna=False).size().reset_index(name="group_size")
duplicate_groups = feature_group_sizes[feature_group_sizes["group_size"] > 1].sort_values("group_size", ascending=False)

print("\nNumber of duplicate feature groups:", len(duplicate_groups))
display(duplicate_groups.head(10))

target_per_feature_group = train.groupby(feature_cols, dropna=False)[TARGET_COL].nunique().reset_index(name="n_targets")
conflicting_groups = target_per_feature_group[target_per_feature_group["n_targets"] > 1]
print("Conflicting duplicate groups with more than one target:", len(conflicting_groups))

In [ ]:
train["feature_group_id"] = pd.util.hash_pandas_object(train[feature_cols], index=False).astype("int64")

group_counts = train["feature_group_id"].value_counts()
print("Number of unique feature groups:", train["feature_group_id"].nunique())
print("Rows belonging to duplicate groups:", int(group_counts[group_counts > 1].sum()))
print("Largest duplicate group size:", int(group_counts.max()))


In [ ]:
if test is not None:
    print("Test rows:", len(test))
    print("Unique test IDs:", test[ID_COL].nunique())
    print("Duplicated test IDs:", test[ID_COL].duplicated().sum())

    test_full_duplicate_rows = test.duplicated().sum()
    test_duplicate_feature_rows = test.duplicated(subset=feature_cols).sum()

    print("\nFull duplicate rows in test:", test_full_duplicate_rows)
    print("Duplicate feature rows in test ignoring ID:", test_duplicate_feature_rows)

    test_group_sizes = (
        test.groupby(feature_cols, dropna=False)
        .size()
        .reset_index(name="group_size")
    )

    test_duplicate_groups = (
        test_group_sizes[test_group_sizes["group_size"] > 1]
        .sort_values("group_size", ascending=False)
    )

    print("Number of duplicate feature groups in test:", len(test_duplicate_groups))
    print("Rows belonging to duplicate feature groups in test:",
          int(test_duplicate_groups["group_size"].sum()) if len(test_duplicate_groups) else 0)

    display(test_duplicate_groups.head(20))

In [ ]:
X_raw = train[feature_cols].copy()
y = train[TARGET_COL].copy()
if test is not None:
    train_keys = pd.util.hash_pandas_object(train[feature_cols], index=False).astype(str)
    test_keys = pd.util.hash_pandas_object(test[feature_cols], index=False).astype(str)

    train_key_target = (
        pd.DataFrame({"key": train_keys, TARGET_COL: y})
        .groupby("key")[TARGET_COL]
        .agg(n_targets="nunique", target_mode=lambda s: s.mode().iloc[0], train_count="size")
        .reset_index()
    )

    matched = pd.DataFrame({
        ID_COL: test[ID_COL],
        "key": test_keys
    }).merge(train_key_target, on="key", how="left")

    print("Test rows exactly matching at least one train feature row:",
          matched["target_mode"].notna().sum())

    print("Matched rows with conflicting train targets:",
          matched["n_targets"].gt(1).sum())

    display(matched[matched["target_mode"].notna()].head(20))

In [ ]:
X_raw = train[feature_cols].copy()
y = train[TARGET_COL].copy()

numeric_cols = X_raw.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

binary_cols = []
continuous_cols = []
for col in numeric_cols:
    vals = set(X_raw[col].dropna().unique())
    if vals.issubset({0, 1}) and len(vals) <= 2:
        binary_cols.append(col)
    else:
        continuous_cols.append(col)

print("Numeric columns:", len(numeric_cols), numeric_cols)
print("Binary columns:", len(binary_cols), binary_cols)
print("Continuous columns:", len(continuous_cols), continuous_cols)
print("Categorical columns:", len(categorical_cols), categorical_cols)

feature_type_table = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(X_raw[c].dtype) for c in feature_cols],
    "n_unique": [X_raw[c].nunique(dropna=True) for c in feature_cols],
    "missing": [X_raw[c].isna().sum() for c in feature_cols],
    "type_used": ["binary" if c in binary_cols else "continuous" if c in continuous_cols else "categorical" for c in feature_cols]
})
display(feature_type_table)

In [ ]:
if continuous_cols:
    display(X_raw[continuous_cols].describe().T)
else:
    print("No continuous features detected.")

In [ ]:
if binary_cols:
    binary_rate_by_target = train.groupby(TARGET_COL)[binary_cols].mean().round(4)
    display(binary_rate_by_target)

    plt.figure(figsize=(12, max(4, len(binary_cols) * 0.5)))
    sns.heatmap(binary_rate_by_target, annot=True, cmap="Blues", fmt=".2f")
    plt.title("Positive rate of binary features by target")
    plt.show()
else:
    print("No binary features detected.")

In [ ]:
if continuous_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(continuous_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, continuous_cols):
        ax.hist(train[col].dropna(), bins=30)
        ax.set_title(col)
        ax.set_xlabel("value")
        ax.set_ylabel("count")

    for ax in axes[len(continuous_cols):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No continuous features to plot.")

In [ ]:
outlier_rows = []
for col in continuous_cols:
    s = train[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (train[col] < lower) | (train[col] > upper)
    outlier_rows.append({
        "feature": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum()),
        "outlier_percent": round(mask.mean() * 100, 2),
        "min": train[col].min(),
        "max": train[col].max(),
        "p99": train[col].quantile(0.99)
    })

outlier_report = pd.DataFrame(outlier_rows).sort_values("outlier_percent", ascending=False)
display(outlier_report)

In [ ]:
outlier_by_target_tables = []
for col in continuous_cols:
    s = train[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    out_mask = ((train[col] < lower) | (train[col] > upper))
    temp = train.assign(is_outlier=out_mask).groupby(TARGET_COL)["is_outlier"].agg(["sum", "mean", "count"])
    temp["feature"] = col
    temp["outlier_rate_percent"] = (temp["mean"] * 100).round(2)
    outlier_by_target_tables.append(temp.reset_index())

if outlier_by_target_tables:
    outliers_by_target = pd.concat(outlier_by_target_tables, ignore_index=True)
    outliers_by_target = outliers_by_target[["feature", TARGET_COL, "sum", "count", "outlier_rate_percent"]]
    display(outliers_by_target.sort_values(["feature", "outlier_rate_percent"], ascending=[True, False]))
else:
    print("No continuous features to analyze.")

In [ ]:
for col in continuous_cols:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=train, x=TARGET_COL, y=col)
    plt.title(f"{col} distribution by target")
    plt.show()

In [ ]:
if len(continuous_cols) >= 2:
    corr = train[continuous_cols].corr()
    display(corr.round(3))

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title("Correlation matrix for continuous features")
    plt.show()
else:
    print("Not enough continuous features for correlation matrix.")

In [ ]:
print("EDA Observations")
print("=" * 60)
print(f"Rows: {len(train)}")
print(f"Features: {len(feature_cols)}")
print(f"Classes: {train[TARGET_COL].nunique()} -> {list(train[TARGET_COL].unique())}")
print("\nTarget distribution:")
print(target_summary)

print("\nMissing values:")
print(f"Total missing in train features: {int(train[feature_cols].isna().sum().sum())}")

print("\nDuplicate feature groups:")
print(f"Duplicate feature rows ignoring ID and target: {duplicate_feature_rows}")
print(f"Conflicting duplicate groups with different target: {len(conflicting_groups)}")

print("\nFeature types:")
print(f"Binary: {binary_cols}")
print(f"Continuous: {continuous_cols}")
print(f"Categorical: {categorical_cols}")

if not outlier_report.empty:
    print("\nTop continuous features by IQR outlier percent:")
    display(outlier_report[["feature", "outlier_count", "outlier_percent", "max", "p99"]].head(10))


# **preproccing**

In [ ]:
X = train[feature_cols].copy()
y = train[TARGET_COL].copy()

test_X = None
if test is not None:
    test_X = test[feature_cols].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("test_X shape:", None if test_X is None else test_X.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape, "X_val:", X_val.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(4))
print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True).round(4))

In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold


groups = pd.util.hash_pandas_object(X, index=False).astype(str)

print("Number of rows:", len(X))
print("Number of unique feature groups:", groups.nunique())
print("Number of duplicated feature groups:", len(X) - groups.nunique())


# cv_regular = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=SEED
# )

cv_grouped = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [ ]:
from sklearn import set_config
set_config(transform_output="default")

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

continuous_tree_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

continuous_standard_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

continuous_robust_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

# categorical_transformer = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="most_frequent")),
#     ("onehot", OneHotEncoder(handle_unknown="ignore"))
# ])


def make_preprocessor(continuous_transformer):
    transformers = []

    if continuous_cols:
        transformers.append(("continuous", continuous_transformer, continuous_cols))

    if binary_cols:
        transformers.append(("binary", binary_transformer, binary_cols))


    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


preprocess_tree = make_preprocessor(continuous_tree_transformer)
preprocess_standard = make_preprocessor(continuous_standard_transformer)
preprocess_robust = make_preprocessor(continuous_robust_transformer)


all_features_standard_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess_all_standard = ColumnTransformer(
    transformers=[
        ("all_features", all_features_standard_transformer, feature_cols)
    ],
    remainder="drop"
)

print("Preprocessors are ready.")

# <div dir="rtl" style="text-align:right">

## experiments classification
</div>

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_val)

print("Dummy Accuracy:", accuracy_score(y_val, dummy_preds))
print("Dummy Macro F1:", f1_score(y_val, dummy_preds, average="macro"))
print("Dummy Weighted F1:", f1_score(y_val, dummy_preds, average="weighted"))
print("\nClassification report:")
print(classification_report(y_val, dummy_preds))

In [ ]:
from sklearn.model_selection import cross_validate
import numpy as np

def evaluate_cv(model_name, pipeline, X, y, cv, groups=None):
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        groups=groups,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro",
            "f1_weighted": "f1_weighted"
        },
        return_train_score=True,
        n_jobs=-1
    )

    return {
        "model": model_name,
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_std": scores["test_accuracy"].std(),
        "cv_f1_macro_mean": scores["test_f1_macro"].mean(),
        "cv_f1_macro_std": scores["test_f1_macro"].std(),
        "cv_f1_weighted_mean": scores["test_f1_weighted"].mean(),
        "cv_f1_weighted_std": scores["test_f1_weighted"].std(),
        "train_f1_macro_mean": scores["train_f1_macro"].mean()
    }

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedGroupKFold
experiments = []

pipe_rf_original = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
])
experiments.append(("RF_original_like_without_class_weight_n_estimators200", pipe_rf_original, {
    "model_type": "RandomForestClassifier",
    "n_estimators": 200,
    "class_weight": "None",
    "preprocessing": "tree_preprocessing_no_scaling"
}))




pipe_rf_balanced_nEstimators200 = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1))
])
experiments.append(("RF_balanced_n_estimators200", pipe_rf_balanced_nEstimators200, {
    "model_type": "RandomForestClassifier",
    "n_estimators": 200,
    "class_weight": "balanced",
    "preprocessing": "tree_preprocessing_no_scaling"
}))



pipe_rf_balanced_nEstimators100 = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1))
])
experiments.append(("RF_balanced_n_estimators100", pipe_rf_balanced_nEstimators100, {
    "model_type": "RandomForestClassifier",
    "n_estimators": 100,
    "class_weight": "balanced",
    "preprocessing": "tree_preprocessing_no_scaling"
}))


pipe_rf_leaf2 = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1
    ))
])
experiments.append(("RF_200_balanced_leaf2_sqrt", pipe_rf_leaf2, {
    "model_type": "RandomForestClassifier",
    "n_estimators": 200,
    "class_weight": "balanced",
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "preprocessing": "tree_preprocessing_no_scaling"
}))




pipe_rf_balanced_subsample_100 = Pipeline(steps=[
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1
    ))
])
experiments.append(("RF_100_balanced_subsample", pipe_rf_balanced_subsample_100, {
    "model_type": "RandomForestClassifier",
    "n_estimators": 100,
    "class_weight": "balanced_subsample",
    "preprocessing": "tree_preprocessing_no_scaling"
}))



# _________________________________________________________________________

pipe_logreg_balanced = Pipeline(steps=[
    ("preprocess", preprocess_standard),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED))
])
experiments.append(("LogReg_standard_balanced", pipe_logreg_balanced, {
    "model_type": "LogisticRegression",
    "max_iter": 3000,
    "class_weight": "balanced",
    "preprocessing": "median_impute_standard_scaler_for_continuous"
}))
pipe_logreg_balanced = Pipeline(steps=[
    ("preprocess", preprocess_robust),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED))
])
experiments.append(("LogReg_robust_balanced", pipe_logreg_balanced, {
    "model_type": "LogisticRegression",
    "max_iter": 3000,
    "class_weight": "balanced",
    "preprocessing": "median_impute_robust_scaler_for_continuous"
}))
# # ________________________________________________________________________________________________




# for C in [0.03, 0.1, 0.3, 1.0]:
pipe_poly_logreg = Pipeline(steps=[
    ("preprocess", preprocess_all_standard),
    ("poly", PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ("model", LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=5000,
        class_weight="balanced",
        random_state=SEED
    ))
])
name = f"Poly2_LogReg_C1.0"
experiments.append((name, pipe_poly_logreg, {
    "model_type": "PolynomialFeatures + LogisticRegression",
    "degree": 2,
    "C": 1.0,
    "penalty": "l2",
    "class_weight": "balanced",
    "preprocessing": "standard_scaler_all_features"
}))


def get_cv(seed):
    return StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

# @@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
SEEDS = [42, 123, 2024, 777, 999]

all_results = []

for name, pipe, params in experiments:
    print(f"\nModel: {name}")

    seed_val_scores = []
    seed_train_scores = []

    for seed in SEEDS:
        cv = get_cv(seed)

        result = evaluate_cv(
            model_name=f"{name}_seed{seed}",
            pipeline=pipe,
            X=X,
            y=y,
            cv=cv,
            groups=groups
        )

        seed_val_scores.append(result["cv_f1_macro_mean"])
        seed_train_scores.append(result["train_f1_macro_mean"])

    val_mean = np.mean(seed_val_scores)
    val_std = np.std(seed_val_scores, ddof=1)
    train_mean = np.mean(seed_train_scores)
    all_results.append({
        "model": name,
        "mean_cv_over_seeds": val_mean,
        "std_cv_over_seeds": val_std,
        "worst_seed_cv": np.min(seed_val_scores),
        "train_cv_over_seeds": train_mean,
        "generalization_gap": train_mean - val_mean,
        "stability_score": val_mean - val_std,
        "all_seed_val_scores": seed_val_scores,
        "all_seed_train_scores": seed_train_scores,
    })

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(
    by=[
        "stability_score",
        "worst_seed_cv",
        "mean_cv_over_seeds"
    ],
    ascending=False
).reset_index(drop=True)

display(results_df)

In [ ]:
import tempfile
import os
import joblib
import mlflow
import numpy as np
from mlflow.models.signature import infer_signature

mlflow.set_tracking_uri("sqlite:///mlflow.db")  # stored in the project root
mlflow.set_experiment("fite-classification-final")


for name, pipe, params in experiments:

    print("Logging:", name)
    matched_rows = results_df[results_df["model"] == name]
    if matched_rows.empty:
        print(f" Skipping {name}: no matching row found in results_df")
        continue

    result_row = matched_rows.iloc[0]

    with mlflow.start_run(run_name=name):

        mlflow.log_param("git_commit", GIT_COMMIT)
        mlflow.log_param("dvc_data_pointer", "data/raw.dvc")
        if DVC_DATA_POINTER.exists():
            mlflow.log_artifact(str(DVC_DATA_POINTER), artifact_path="data_version")

        mlflow.log_params(params)
        mlflow.log_param("n_seeds", len(SEEDS))
        mlflow.log_param("seeds", str(SEEDS))
        mlflow.log_param("n_features", len(feature_cols))
        mlflow.log_param("n_classes", y.nunique())
        mlflow.log_param("binary_cols", ",".join(map(str, binary_cols)))
        mlflow.log_param("continuous_cols", ",".join(map(str, continuous_cols)))

        if "categorical_cols" in globals():
            mlflow.log_param("categorical_cols", ",".join(map(str, categorical_cols)))
        else:
            mlflow.log_param("categorical_cols", "")

        mlflow.log_metric("cv_f1_macro_mean", float(result_row["mean_cv_over_seeds"]))
        mlflow.log_metric("cv_f1_macro_std", float(result_row["std_cv_over_seeds"]))
        mlflow.log_metric("cv_f1_macro_worst_seed", float(result_row["worst_seed_cv"]))

        mlflow.log_metric("train_f1_macro_mean", float(result_row["train_cv_over_seeds"]))
        mlflow.log_metric("generalization_gap", float(result_row["generalization_gap"]))
        mlflow.log_metric("stability_score", float(result_row["stability_score"]))


        for i, score in enumerate(result_row["all_seed_val_scores"]):
            mlflow.log_metric(f"cv_val_seed_{i}", float(score))

        for i, score in enumerate(result_row["all_seed_train_scores"]):
            mlflow.log_metric(f"cv_train_seed_{i}", float(score))


        pipe.fit(X, y)
        input_example = X.head(5)
        signature = infer_signature(input_example, pipe.predict(input_example))

        try:
            mlflow.sklearn.log_model(
                sk_model=pipe,
                artifact_path="model_pipeline",
                serialization_format="cloudpickle",
                input_example=input_example,
                signature=signature
            )

        except Exception as e:
            print("MLflow sklearn failed → fallback joblib")
            print(e)

            with tempfile.TemporaryDirectory() as tmpdir:
                model_path = os.path.join(tmpdir, "model.joblib")
                joblib.dump(pipe, model_path)
                mlflow.log_artifact(model_path, artifact_path="model_joblib")

print(" MLflow logging completed")

In [ ]:
best_model_name = results_df.iloc[0]["model"]
print("Best model by CV macro F1:", best_model_name)

best_pipeline = None
for name, pipe, params in experiments:
    if name == best_model_name:
        best_pipeline = pipe
        best_params = params
        break

best_pipeline.fit(X, y)
print("Best pipeline fitted on full training data.")
print("Best params:", best_params)

In [ ]:
if test is not None:
    test_predictions = best_pipeline.predict(test_X)
    submission = pd.DataFrame({
        ID_COL: test[ID_COL],
        TARGET_COL: test_predictions
    })

    display(submission.head())
    print("Submission shape:", submission.shape)
    print("Columns:", list(submission.columns))
    print("Predicted target distribution:")
    print(submission[TARGET_COL].value_counts(normalize=True).round(4))

    output_path = SUBMISSION_DIR / f"submission_{best_model_name}.csv"
    submission.to_csv(output_path, index=False)
    print("Saved submission to:", output_path)
else:
    print("test_data.csv not found. Submission file was not created.")

In [ ]:
print("To open the MLflow dashboard, run this in the VS Code terminal from the project root:")
print("mlflow ui --backend-store-uri sqlite:///mlflow.db")
print("Then open: http://127.0.0.1:5000")